# Preprocessing, Dataset 1: ULB Credit Card Fraud (creditcard.csv)

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = '../Dataset/creditcardfraud/creditcard.csv'
OUT_PATH = '../Dataset/processed/creditcard_clean.csv'

In [2]:
df = pd.read_csv(DATA_PATH)
print("Starting shape:", df.shape)
print("Starting class balance:")
print(df['Class'].value_counts())

Starting shape: (284807, 31)
Starting class balance:
Class
0    284315
1       492
Name: count, dtype: int64


## Drop exact duplicates

In [3]:
# found in EDA, 1081 exact duplicate rows, drop before anything else so they can't leak across
# whatever split/CV happens downstream
n_dupes = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {n_dupes} duplicate rows")
print("Shape after dedup:", df.shape)
print("Class balance after dedup:")
print(df['Class'].value_counts())

Dropped 1081 duplicate rows
Shape after dedup: (283726, 31)
Class balance after dedup:
Class
0    283253
1       473
Name: count, dtype: int64


## Feature engineering

In [4]:
# Time is just seconds elapsed since the first transaction in this 2-day capture window, not
# meaningful on its own, hour of day is the generalisable part of it
df['hour_of_day'] = (df['Time'] % 86400 // 3600).astype(int)
df = df.drop(columns=['Time'])
df[['hour_of_day']].describe()

,hour_of_day
count,283726.000000
mean,14.045646
std,5.834817
min,0.000000
25%,10.000000
50%,15.000000
75%,19.000000
max,23.000000


Second dedup pass: collapsing `Time` down to `hour_of_day` means rows that were only distinct by their exact second now collide. Same leakage logic as the first dedup, applied again after feature engineering rather than before.

In [5]:
n_dupes_post_fe = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {n_dupes_post_fe} duplicate rows created by the hour_of_day collapse")
print("Shape after second dedup:", df.shape)
print("Class balance after second dedup:")
print(df['Class'].value_counts())

Dropped 2804 duplicate rows created by the hour_of_day collapse
Shape after second dedup: (280922, 31)
Class balance after second dedup:
Class
0    280449
1       473
Name: count, dtype: int64


No scaling here, `V1`-`V28` are already PCA components, and `Amount`/`hour_of_day`
get scaled inside the Logistic Regression pipeline at modelling time, not baked into a
file Random Forest also reads.

## Final checks and save

In [6]:
print("Missing values:", df.isna().sum().sum())
print("Final shape:", df.shape)
print("Feature columns:", [c for c in df.columns if c != 'Class'])
print("Feature count:", df.shape[1] - 1)

Missing values: 0
Final shape: (280922, 31)
Feature columns: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'hour_of_day']
Feature count: 30


In [7]:
df.to_csv(OUT_PATH, index=False)
print("Saved to", OUT_PATH)

Saved to ../Dataset/processed/creditcard_clean.csv


## Summary

In [8]:
fraud_pct = df['Class'].value_counts(normalize=True) * 100
print(f"Rows: {df.shape[0]}, Feature cols: {df.shape[1] - 1}")
print(f"Fraud rate: {fraud_pct[1]:.3f}%")
print(f"Missing values: {df.isna().sum().sum()}")

Rows: 280922, Feature cols: 30
Fraud rate: 0.168%
Missing values: 0
